<div class="sam-kicker">Princeton · Prepared for Sam · November 16, 2026</div>
<div class="sam-title">Alaska 2026: voter distributions across three candidate axes</div>

**Rachel’s November 11 CSV snapshot · Alaska counting rules · original one-dimensional method**

This notebook brings the completed descriptive results together with Governor/House candidate axes, Mahshad-style voter distributions, permutation-based confusion scenarios, and a dedicated analysis of **Eric Hafner’s House ballots in section 7**. It includes both presentation figures with Sam’s requested labels and stacking order.

- The completed Senate result is unchanged: **gamma is 92.76% across nonempty named prefixes, or 61.54% among three-/four-name prefixes**; the matched random means are **89.00%** and **41.57%**.
- New three-/four-name consistency is **57.23% Governor versus 50.51% random**, and **65.27% House versus 41.26% random**.
- In the hypothetical replay of **3,940 Dan J.–first ballots**, **1,333 reach Peltola (33.83%)**, **904 reach Daniel S. (22.94%)**, and **1,703 become inactive (43.22%)**. Peltola receives **1,147 directly after Dan J.’s elimination and 186 through Heikes**. Both finalists’ routes are shown explicitly in section 5; the rankings do not identify confusion.
- The **2,468** Daniel-first ballots with no countable incumbent preference include **765 with a countable Peltola preference** and **1,703 with neither finalist countable**. Treating every one as intended incumbent support is a strong conditional scenario, not an estimate of confusion.

<div class="sam-note"><b>Reading the voter curves:</b> rankings generally identify intervals, not unique voter locations. The curves use Mahshad’s allocation convention and show only placeable ballots. Singleton-pinning and fitting-budget sensitivities, full cohort coverage, and all exclusions are reported. The three elections have separate coordinate systems.</div>

In [ ]:
from pathlib import Path
import sys,json,csv,hashlib,contextlib,io,os
import numpy as np
import pandas as pd
from IPython.display import display,HTML,Image
ROOT=Path.cwd().resolve()
assert (ROOT/'inputs/Alaska_20261103_US_Senate_11_11.csv').is_file(), 'Open this notebook from the delivered package folder.'
for folder in [ROOT,ROOT/'common',ROOT/'axis_models',ROOT/'voter_distributions',ROOT/'confusion',ROOT/'validation',ROOT/'presentation_figures']:
    sys.path.insert(0,str(folder))
RACHEL_INPUT_READS=set()
def source_guard(event,args):
    if event!='open' or not args or not isinstance(args[0],(str,bytes,os.PathLike)):return
    try:p=Path(os.fsdecode(args[0])).resolve()
    except (ValueError,OSError):return
    mode=args[1] if len(args)>1 else None
    reading=not isinstance(mode,str) or 'r' in mode
    if ROOT.parent in p.parents and ROOT not in p.parents and p.suffix.lower() in {'.csv','.tsv','.gz','.zip','.json','.sqlite','.xlsx'} and reading:
        raise PermissionError('Rachel-only analysis: external workspace data access blocked: '+str(p))
    if ROOT/'inputs' in p.parents and p.suffix.lower()=='.csv' and reading:RACHEL_INPUT_READS.add(p.name)
sys.addaudithook(source_guard)
import presentation as show
from data import RACES,profile,sha256
show.theme()


## 1 · Completed descriptive questions

The earlier descriptive questions are retained and revalidated, rather than treated as new analyses. The main cohorts use **literal rank 1** in the Senate and in the Governor/House first-choice tables. They include **117,437 Peltola**, **115,849 incumbent Dan S. Sullivan**, **5,054 Heikes**, and **3,940 outsider Daniel J. Sullivan Jr.** ballots.

**568 Daniel-first ballots rank both finalists countably, with Peltola above Dan S. Sullivan (14.42%).** The reverse is **423 (10.74%)**. Keeping the original cohort fixed makes the comparison explicit. A ballot naming only Peltola in its countable portion is shown separately from a ballot ranking both finalists.

In [ ]:
import verify_completed
with contextlib.redirect_stdout(io.StringIO()):completed_check=verify_completed.main()
assert completed_check['status']=='passed'
show.first_choice_comparison()


### Governor and House first choices

Affiliation refers to the **chosen candidate or ticket**, not voter party registration. Rachel’s files contain no voter-registration party field. Original blank first ranks, overvotes, write-ins, and absent Governor rows remain in the cohort denominator. The Governor ticket labels include the Democrat/Nonpartisan Kreiss-Tomkins/Johnson ticket; House nonpartisan is Bill Hill.

The first figure presents the four cohorts equally spaced. The second places the same paired bars at their actual Senate MDS coordinates. Both include the presentation edits: **Governor first choice**, **House first choice**, **Senate 1st choice:**, and Libertarian below Republican in the House stacks.

In [ ]:
import build_vertical,build_axis
with contextlib.redirect_stdout(io.StringIO()):
    build_vertical.main()
    build_axis.main()
show.image('presentation_figures/Alaska_Governor_House_First_Choice_Presentation.png',width=1000)
show.image('presentation_figures/Alaska_Candidate_Axis_First_Choice_Presentation.png',width=1250)


### Complete first-choice tables

Each cell is ballots and the percentage of the **full Senate first-choice cohort**. These tables retain all recorded categories, including Governor-file absences.

In [ ]:
show.cross_table('governor')
show.cross_table('us_house')
show.cross_table('governor',party=True)
show.cross_table('us_house',party=True)

## 2 · Senate Sankey and actual transfers

The completed Sankey retains **all 244,705 Senate records** through its four display stages: initial allocation, generic write-in exclusion, Daniel J. elimination, and Heikes elimination. It follows countable preferences. Candidate order and flow conservation have been rechecked from Rachel’s source rows.

Daniel’s **literal first-choice cohort is 3,940**. His **first-structurally-usable-choice cohort is 3,992**; after write-in transfers, his replay node is **4,055**. Those are different denominators. The diagram’s initial write-in stage is explanatory; unresolved generic write-ins are excluded from the four-candidate named model.

Final replay counts remain **119,933 Peltola**, **119,223 Dan S. Sullivan**, and **5,549 inactive**. There are **1,363 initially inactive**, so **4,186 become newly inactive** relative to the graphic’s initial stage. This is a preliminary snapshot replay, not a certified statewide result.

In [ ]:
show.image('completed/flows/senate_sankey.png',width=1250)

## 3 · The original one-axis model in all three elections

Each election is fitted on its **full nonempty countable named profile** and scored on that profile. Only explicitly ranked candidates constrain a ballot; unranked candidates are not silently appended. One-/two-name prefixes pass automatically, so overall gamma and the more informative three-/four-name gamma are shown separately.

The Senate fit and its 49 randomized fits were already completed and independently audited; they are carried forward unchanged after verifying that the new package’s source profile matches exactly. Governor and House are newly fitted using the same **20 outer runs × 8 starts**, 1,000-iteration budget and 10⁻⁶ tolerance. Each of their **49 matched-tail randomized profiles is independently refitted and scored**, preserving first named choice × prefix length.

The displayed House axis is reflected for readability; raw coordinates remain saved. Reflection changes neither gamma nor interval feasibility. Separate normalization to [0,1] does **not** make the three axes a shared ideology scale.

In [ ]:
import run_axes,audit_axes
# The unchanged, independently audited spatial fits are supplied with this notebook.
# Set True only to repeat the expensive Governor/House observed and 98 null fits.
REFIT_NEW_AXES=False
if REFIT_NEW_AXES:
    with contextlib.redirect_stdout(io.StringIO()):
        newly_fitted=run_axes.main()
        axis_audit=audit_axes.main()
else:
    axis_audit=show.load('axis_models/independent_axis_audit.json')
assert axis_audit['status']=='passed'
senate_profile,senate_source=profile('us_senate')
saved=show.load('axis_models/us_senate/profile.json')
assert senate_profile=={tuple(row['ranking']):row['count'] for row in saved['ballots']}
assert senate_source['source_sha256']==show.load('axis_models/us_senate/mds_results.json')['source_sha256']
show.axis_summary()
show.plot_axes()


## 4 · Voter distributions on the candidate axes

**Mahshad’s plotting convention.** For a compatible multi-name ranking, intersect the ranked-pair distance constraints to obtain a possible interval on the candidate axis. A pattern occurring *n* times contributes *n* equally spaced locations across that interval, as in the original `np.linspace` procedure. A one-name ballot is placed at that candidate’s position. These are **imputed locations**, not directly measured voter positions.

Each curve is a boundary-corrected Gaussian density of those finite allocated points. Cohorts share a bandwidth within each election, derived from all allocated points using their actual repeated-point count. Reflection at the 0/1 boundaries is a display convention. The exact point arrays, alternative group-specific bandwidths, numerical approximation checks, and interval bounds are included. This compares cohort distributions; the synthetic positions within a repeated ranking pattern are not uniquely assigned to individual ballot IDs, and no individual cross-election location correlation is estimated.

Missing contest rows, no usable named choice, and inconsistent rankings remain separately counted. An inconsistent ranking cannot be placed exactly on the fitted line and is excluded from these curves, rather than forced to a convenient location. Every displayed group’s curve integrates to one **among its plotted ballots**; full-cohort mass-scaled densities are also supplied.

In [ ]:
import run_distributions
with contextlib.redirect_stdout(io.StringIO()):voter_results=run_distributions.main()
assert voter_results['status']=='passed'
show.image('voter_distributions/all_voters_three_axes.png',width=1150)


### Peltola and challenger Sullivan across the three elections

Here “Peltola voters” always means the **same 117,437 literal Senate Peltola-first ballots**. “Challenger Sullivan” means the **same 3,940 literal Senate Daniel J. Sullivan Jr.-first ballots**, not incumbent Dan S. Sullivan. Their Governor and House rankings are linked by normalized ballot ID.

Under the main allocation convention, Peltola’s group has imputed medians at **Peltola on the Senate line**, **Kreiss-Tomkins on the Governor line**, and **Hill on the House line**. Daniel’s group is spread much more broadly across the Governor and House lines. These summaries are conditional on placement and strongly influenced by the one-name allocation convention; they are not measurements on a common political scale.

In [ ]:
show.image('voter_distributions/peltola_challenger_across_three_axes.png',width=1250)
show.distribution_coverage(focal=True)
show.focal_position_summaries()

### All four Senate first-choice groups on each election’s line

Each panel uses the named candidates of that election. The colored curves refer to Senate first-choice groups throughout. Comparisons within a panel share a smoothing bandwidth. The coverage figure accounts for each group’s full cohort, including ballots absent from a particular contest file.

In [ ]:
show.image('voter_distributions/senate_groups_across_three_axes.png',width=1150)
show.image('voter_distributions/voter_distribution_coverage.png',width=1150)

### Placement and fitting sensitivities

A one-name ranking alone imposes no ranked-pair restriction: every point in [0,1] is feasible under this consistency definition. Pinning it at the candidate is Mahshad’s convention. The singleton-uniform sensitivity shows how the curves change if those ballots are spread across their full feasible interval instead. Large changes reveal assumption dependence.

A stronger **100 × 16** observed fitting budget preserves each election’s candidate order and gamma but shifts some coordinates. The saved fitting-budget sensitivity recalculates all group placements under those stronger-fit maps; the main plots retain the prespecified 20 × 8 maps. Moving point masses can produce large empirical-CDF differences, so those differences are not an ideological-location error measure.

In [ ]:
show.image('voter_distributions/singleton_allocation_sensitivity.png',width=1150)
geometry=pd.read_csv(ROOT/'voter_distributions/axis_geometry_sensitivity.csv')
show.table(geometry[['contest','group_name','main_plotted_n','strong_plotted_n','main_imputed_mean','strong_imputed_mean','main_imputed_median','strong_imputed_median']].round(4),'Main versus stronger-fit geometry; allocation-dependent coordinates')
show.distribution_coverage()


## 5 · Where do Dan J.–first ballots go?

### Support reaching Daniel S., the incumbent

Among the same **3,940 ballots that recorded Dan J. first**, **904 would reach Daniel S. (22.94%)** in our hypothetical replay: **638 directly after Dan J.’s elimination**, and **266 through Heikes**. These are fallback preferences recorded on the ballots. The rankings do not establish that the original first choice was a name-confusion error.

Examples are `Dan J. → Daniel S.` and `Dan J. → Heikes → Daniel S.`. If a continuing Peltola appears before Daniel S., the ballot stays with Peltola; a lower Daniel S. ranking does not jump over her. Thus the **1,472 ballots with both Sullivans usable are not all incumbent transfers**: **568 put Peltola above Daniel S.**

On the separate **4,055-ballot Dan J. replay allocation**, **933** would eventually reach Daniel S. (**655 direct + 278 through Heikes**). That allocation includes incoming ballots from other first choices. **904** remains the answer for the original **3,940-ballot** cohort.

In [ ]:
import run_confusion,verify_confusion
with contextlib.redirect_stdout(io.StringIO()):
    confusion=run_confusion.main()
    confusion_audit=verify_confusion.main()
assert confusion['status']=='passed'
show.routes_and_omissions()


### Support reaching Peltola

**Peltola would receive more of these Dan J.–first ballots than Daniel S. would:**

- **1,147** would transfer directly to Peltola after Dan J. is eliminated.
- **186** would first count for Heikes, then transfer to Peltola when Heikes is eliminated.
- **Total: 1,333 reach Peltola (33.83%)**, compared with **904 reaching Daniel S. (22.94%)**. Another **1,703 become inactive (43.22%)** because no usable preference remains for either finalist.

These are **hypothetical final destinations**, assuming Dan J. and then Heikes are eliminated while Peltola and Daniel S. remain. They use the same Alaska counting rules and the same **3,940 literal first-choice ballots** as the incumbent analysis, rather than the larger Sankey allocation.

The **1,333** comprise **568** ballots that validly rank both finalists with Peltola ahead, plus **765** with a usable Peltola preference and no usable Daniel S. preference. These are two separate categories. The transfer pattern documents Peltola’s recorded fallback support; it does not establish why voters chose Dan J. first.

In [ ]:
import verify_peltola_routes
with contextlib.redirect_stdout(io.StringIO()):peltola_route_audit=verify_peltola_routes.main()
assert peltola_route_audit['status']=='passed'
show.finalist_route_comparison()


### Which permutations might reflect confusion?

An outsider-first ranking followed by the incumbent, or an outsider-first ranking omitting the incumbent, is **compatible with** a name-confusion explanation. Both can also be sincere preferences. Adjacent Sullivans, a Heikes intermediary, or an omitted finalist are observable pattern definitions; none identifies intent by itself.

The complete permutation table retains all 16 countable named patterns beginning with Daniel. Pattern sets overlap where stated. Counts of “could be confused” are ceilings within defined sets, not observed counts of confused people. A protest-vote interpretation is likewise not established by these ballots.

### All Dan J.–first permutations

These are countable unique-name permutations, with write-in removed only after applying overvote/skip barriers. Original five-cell rank strings and all annotations remain in the source files and detailed exports.

In [ ]:
show.permutation_table()

## 6 · Possible lost support: omissions and conditional ceilings

**No countable incumbent preference:** **2,468** literal Daniel-first ballots, split into **765 with countable Peltola** and **1,703 with neither finalist countable**. This is the clean Alaska-rule answer for support unavailable to the incumbent.

**“Not ranked at all” needs a source qualification.** There are **2,462 with no explicit incumbent name in the five coded cells**. Of these, **2,441 have no overvote token** and **21 contain generic overvotes whose candidate membership is unknown**. Another **six visibly name the incumbent only beyond a counting barrier**. We do not fill hidden identities from the old CVR.

Within the 2,462 visibly-incumbent-absent ballots, **777 visibly name Peltola**, while **765 have Peltola in the countable portion**. Within the narrower 2,441 no-overvote ballots, **760 have countable Peltola** and **1,681 become inactive**. Original visible marks are used here only to distinguish what “not ranked anywhere” can mean; the main transfer counts use legal availability.

**Conditional ceiling:** assume every ballot in a chosen set intended incumbent Dan first, correct those ballots to him, and leave all other ballots fixed. For the 2,468 no-countable-incumbent set, the ceiling is **2,468 additional incumbent votes**. The Dan-minus-Peltola margin could increase by **3,233**: twice the 765 switches from Peltola, plus the 1,703 inactive recoveries. Actual confusion could be zero. These assumptions are stronger than anything the rankings establish.

The broader all-Daniel-first scenario has a ceiling of **3,036 additional incumbent votes** and **4,369 in margin change**, because 904 already reach the incumbent. These scenarios are **not added together**, do not estimate actual confusion, and do not bound every conceivable confusion mechanism in the election.

In [ ]:
show.upper_bounds()

### Identified ballot examples

Sam’s “ballot #34” is treated as an illustration because no corresponding source ID was supplied. The examples below instead use actual anonymous Rachel ballot IDs and show their source rank cells and replay path. A source row index is explicitly zero-based; it is not a voter identity or a claim to have identified Sam’s #34.

In [ ]:
examples=pd.read_csv(ROOT/'confusion/anonymous_example_ballots.csv').head(5)
show.table(examples[['illustration','source_ballotID','rank1','rank2','rank3','rank4','rank5','after_Daniel_elimination','after_Heikes_elimination']],'Actual source records illustrating the mechanisms')


## 7 · Eric Hafner’s House ballots

**What did Hafner’s voters actually rank, and how did they vote in the other races?** This section uses only Rachel’s unchanged **November 11** files. The main group consists of ballots that **recorded Eric Hafner in House rank 1**. It does not assume that those voters were confused, uninformed, or voting insincerely.

**Background.** The [U.S. Department of Justice](https://www.justice.gov/usao-nj/pr/monmouth-county-man-sentenced-20-years-prison-making-threatening-communications-and) announced that Hafner was sentenced to **20 years in federal prison on December 7, 2023**, for threatening communications and false bomb threats. [Alaska’s News Source reported in June 2026](https://www.alaskasnewssource.com/2026/06/06/prisoner-his-mom-how-are-candidates-whove-never-visited-alaska-able-run-states-federal-seats/) that he had never visited Alaska and was serving his sentence in New York. The [official 2026 candidate list](https://www.elections.alaska.gov/election-candidates/?election=26genr) listed him as a registered Democrat. These are background annotations; the linked articles supply no ballot data.

In [ ]:
sys.path.insert(0,str(ROOT/'hafner'))
import run_hafner,audit_hafner
from hafner_notebook import show_summary,show_cross_election,show_details
with contextlib.redirect_stdout(io.StringIO()):
    hafner_results=run_hafner.main()
    hafner_audit=audit_hafner.main()
assert hafner_results['status']=='passed'
assert hafner_audit['status']=='passed'
show_summary(hafner_results)


### Backup rankings and hypothetical House destinations

The left panel shows the **next usable named preference** after Hafner. The right panel shows where the same ballots would end up in the **full-source House replay**.

**McDermott is eliminated before Hafner in this snapshot.** A ballot ranking `Hafner → McDermott → Hill` would therefore move **directly from Hafner to Hill** when Hafner is eliminated: McDermott is already out. A `Hafner → McDermott` ballot with no eligible later preference would become inactive. The listed backup order is not automatically the sequence of actual transfers.

“Inactive” means that no usable preference remains for either continuing finalist. Earlier valid votes still count in earlier rounds. These are hypothetical transfers in a dated snapshot, not official or final statewide results.

In [ ]:
show.image('hafner/hafner_backup_choices_and_destinations.png',width=1150)

### The same voters in the Senate and governor races

These are **linked records from the same ballots**, joined by normalized ballot ID. They show literal first choices in the other contests. All four House first-choice groups are shown for comparison. Candidate affiliations describe the chosen candidates, not voters’ party registration. Missing governor records stay in the denominator.

In [ ]:
show_cross_election(hafner_results)
show.image('hafner/hafner_cross_election_comparison.png',width=1150)

### Where did Hafner receive the most support?

Compare **the number of Hafner-first ballots** with **his share of all House records in an area**. A larger population can produce more votes without producing a higher share. The district chart combines all recorded voting modes; the geographic-precinct table is Election Day only in these files.

Area descriptions follow the source precinct labels and the [Division of Elections’ district references](https://www.elections.alaska.gov/research/district-maps/). The [District 18 precinct maps](https://www.elections.alaska.gov/doc/maps/2023-precincts/HD18_May2023Final_Maps.pdf) show Government Hill, JBER and North Muldoon. State House districts and administrative counting groups must not be treated as exact city boundaries.

In [ ]:
import hafner_geography
hafner_geography_results=hafner_geography.main()
assert hafner_geography_results['status']=='passed'
hafner_geography.show_geography(hafner_geography_results)


### What the ballots establish

Hafner-first voters overwhelmingly also chose Peltola for Senate and Kreiss-Tomkins/Johnson for governor, resembling the cross-race choices of Hill-first voters. At the same time, many Hafner-first ballots contain no usable backup for either House finalist. The tables distinguish strictly blank later ranks from repeats, write-ins, and preferences stopped by counting barriers.

This describes the recorded choices. Explaining **why** people selected Hafner would require evidence about what voters knew or intended, such as a voter survey. These records do not identify party-label voting, protest voting, confusion, awareness of his imprisonment, or the effect of campaign messages. A large tally alone does not establish that the tally is erroneous.

The full **16-row ranking distribution**, original-mark diagnostics, exact other-race counts, and denominator checks are in **“Eric Hafner: all rankings, original marks, and additional checks”** under **Methods, checks, and additional tables**. Machine-readable tables and a source-preserving cohort export are in the package’s `hafner/` folder.

### Eric Hafner: full distributions and checks

All percentages below use the **17,918 literal Hafner-first ballots**, unless the table explicitly gives a different denominator. Generic write-ins retain their rank occupancy while the Alaska overvote/skipped-rank barriers are applied; only then are write-ins removed from the named-candidate sequences. Repeated names are collapsed. Original five-cell patterns and source annotations are preserved separately.

“Next usable named preference” describes the ranking after Hafner; the later replay allocation also depends on which candidates are still running. These definitions must not be interchanged. The 254 original five-cell patterns remain available in `hafner/hafner_raw_five_rank_patterns.csv`; the complete source rows are in `hafner/hafner_literal_first_ballots.csv.gz`.

In [ ]:
show_details(hafner_results)
hafner_geography.show_geography_details(hafner_geography_results)

## 8 · Sources, versioning and reproducibility

All ballot analyses use only the three unmodified Rachel/FairVote **November 11** CSVs, copied under `inputs/` with exact hashes. This remains the selected input version even though later election data may exist. Every original rank cell and metadata column is preserved. Candidate IDs and affiliation labels are reference annotations, not fields invented as Rachel-supplied IDs or voter registration.

Alaska-countable prefixes keep earlier valid choices, stop before an overvote, cross one internal skipped rank, stop at two consecutive internal skips, and collapse repeat candidates. A generic write-in occupies its original rank while barriers are evaluated, then is excluded from the four-candidate models. The [Alaska Division of Elections](https://www.elections.alaska.gov/election-results/e/?id=26genr) documents these rules.

The source files have **244,705 Senate/House** rows and **244,575 Governor** rows; **130 Governor rows are absent** from the federal ID universe. These are scanned CVR records, not all statewide or certified ballots. The archived November 11 FairVote-linked round tables match all 54 available numeric cells; that validation does not turn the scanned snapshot into a certified tally. The older article’s visible-name subgroup comparison remains supplementary and does not enter these main results.

**Run All:** revalidates the completed descriptive/Sankey outputs from source, rebuilds both presentation figures, reconstructs voter placements and densities, recomputes confusion permutations/scenarios, and rebuilds the Hafner analysis with an independent source audit. By default, the unchanged, independently audited spatial fits are reused; set `REFIT_NEW_AXES=True` to repeat Governor/House observed and all 98 null fits. The completed Senate fit is explicitly carried forward and checked against the exact current source profile. Independent audit scripts and machine-readable tables are included. Python 3.11+ is required; the pinned execution environment uses Python 3.12.14.

In [ ]:
assert RACHEL_INPUT_READS=={meta['filename'] for meta in RACES.values()}
source_audit={'source_policy':'Rachel November11CSV inputs only','observed_input_files':sorted(RACHEL_INPUT_READS),
 'old_workspace_ballot_inputs_blocked':True,'completed_results_revalidated':completed_check['status']=='passed',
 'new_Governor_House_models_refitted':REFIT_NEW_AXES,'Senate_model_carried_forward_with_exact_profile_verification':True,
 'unchanged_audited_spatial_models_reused':not REFIT_NEW_AXES,
 'hafner_analysis_recomputed':hafner_results['status']=='passed','independent_hafner_audit':hafner_audit['status'],
 'both_finalists_routes_independently_verified':peltola_route_audit['status']=='passed',
 'hafner_geography_recomputed_and_reconciled':hafner_geography_results['status']=='passed'}
(ROOT/'validation/notebook_source_audit.json').write_text(json.dumps(source_audit,indent=2))
show.table([{'Rachel input':slug,'Rows':completed_check['sources'][slug]['rows'],'SHA-256':meta['sha256']} for slug,meta in RACES.items()],'Exact source versions')
print('All notebook analyses completed using Rachel’s selected source version.')
